# Fractal accelerator — DDR writeback bring-up

Loads the overlay, sets the PL clock, programs a **full Mandelbrot (zoom 0)**, starts the
engine, waits for `all_done`, then watches the **AXI HP writeback** drain into a DDR buffer
and reconstructs the image.

This notebook is the hardware counterpart of the writeback fix: it polls `TRANS_COUNT`
(number of completed AXI HP bursts) so you can *see* the B-channel completing rather than
stalling at 0.

**Before running:** set `BITSTREAM` below to your `.bit` (its `.hwh` must sit next to it).

In [ ]:
# ── 0. Config you may need to edit ───────────────────────────────────────────
BITSTREAM   = "/home/xilinx/pynq/overlays/fractal/fractal.bit"  # <-- your .bit (.hwh beside it)
FCLK0_MHZ   = 100.0      # PL fabric clock fed to the engine AND S_AXI_HP0_ACLK
SLAVE_HINT  = "axi_lite_slave"  # substring used to find the control slave in ip_dict

# Image geometry (fixed by the RTL: one 256x256 sixteenth per engine, 16 sixteenths)
SIXTEENTH_W = 256
SIXTEENTHS  = 16                 # full image is 4x4 sixteenths = 1024x1024
IMAGE_BYTES = SIXTEENTH_W * SIXTEENTH_W * SIXTEENTHS   # 1 byte/pixel

In [ ]:
# ── 1. Register map (byte offsets) — from axi_lite_slave_debug.sv ─────────────
CTRL            = 0x00   # [0]=start pulse, [1]=soft reset (level)
STATUS          = 0x04   # [0]=started [1]=all_done(sticky) [2]=axi_err(sticky)
FRACTAL_TYPE    = 0x08   # [4:0]
CENTRE_X           = 0x0C
CENTRE_Y           = 0x10
ZOOM_LEVEL      = 0x14
MAX_ITER        = 0x18   # [11:0]
IMAGE_BASE_ADDR = 0x1C   # [31:8] used; [7:0] forced 0 (256B align)
IRQ_ENABLE      = 0x20
TRANS_COUNT     = 0x24   # RO: completed AXI HP bursts
CLR_STATUS      = 0x28   # [0]=clr all_done [1]=clr axi_err [2]=clr trans [3]=clr dbg
JULIA_REAL      = 0x2C
JULIA_IMAG      = 0x30
# writeback / compute debug (RO)
DBG_SCHED_STATE = 0x34
DBG_FLAGS       = 0x4C   # [0]=comp_complete [1]=comp_differ [2]=engine_done
DBG_TILE_DONE   = 0x50   # popcount(tile_done)
DBG_B2D_STATE   = 0x54   # [2:0]=b2d fsm [4]=any_pending [8:5]=axi fsm [9]=wr_ready
DBG_AW_HS_CNT   = 0x60
DBG_W_HS_CNT    = 0x64
DBG_B_HS_CNT    = 0x68
DBG_BRESP_LAST  = 0x6C
DBG_XFER_POP    = 0x74   # popcount(transferred)

# CTRL / STATUS bits
CTRL_START      = 1 << 0
CTRL_SOFT_RESET = 1 << 1
ST_STARTED      = 1 << 0
ST_ALL_DONE     = 1 << 1
ST_AXI_ERR      = 1 << 2

In [ ]:
# ── 2. Load overlay + set the PL clock ───────────────────────────────────────
# The same FCLK0 feeds the engine and S_AXI_HP0_ACLK in the BD, so setting it here
# guarantees the HP write port is actually clocked (a dead HP clock is the classic
# cause of AW issued but B never returning).
from pynq import Overlay, allocate, Clocks
import numpy as np, time

ol = Overlay(BITSTREAM)              # downloads the bitstream
Clocks.fclk0_mhz = FCLK0_MHZ         # program PL fabric clock 0
print(f"fclk0 set to {Clocks.fclk0_mhz:.3f} MHz")
print("IP in overlay:", list(ol.ip_dict.keys()))

In [ ]:
# ── 3. Find the control slave from ip_dict (no hardcoded base address) ────────
from pynq import MMIO

slave_key = next((k for k in ol.ip_dict if SLAVE_HINT in k), None)
assert slave_key, f"no IP matching {SLAVE_HINT!r} in {list(ol.ip_dict)}"
entry = ol.ip_dict[slave_key]
base, rng = entry['phys_addr'], entry['addr_range']
ctrl = MMIO(base, rng)
print(f"control slave '{slave_key}'  base=0x{base:08X}  range=0x{rng:X}")

def w(off, val): ctrl.write(off, val & 0xFFFFFFFF)
def r(off):      return ctrl.read(off)

In [ ]:
# ── 4. Allocate the DDR image buffer (HP-DMA target) ─────────────────────────
# allocate() gives a physically-contiguous, cache-coherent buffer. Its physical
# address goes into IMAGE_BASE_ADDR; the HP master writes pixels straight here.
img_buf = allocate(shape=(IMAGE_BYTES,), dtype=np.uint8)
img_buf[:] = 0
img_buf.flush()                       # push the zero-fill out to DRAM
phys = img_buf.physical_address
assert phys % 256 == 0, f"buffer not 256B aligned: 0x{phys:X}"   # RTL forces [7:0]=0
print(f"image buffer  phys=0x{phys:08X}  size={IMAGE_BYTES} bytes")

In [ ]:
# ── 5. Program a full Mandelbrot, zoom 0 ─────────────────────────────────────
# fractal_type 0 = Mandelbrot. zoom 0 = widest view (scale 2^25). pan 0 = centred.
# centre_x/centre_y are fixed-point; the slave appends 3 low zero bits, so write the
# top 32 bits of the 35-bit value. 0 is 0 either way for the centred view.
FRACTAL_MANDELBROT = 0
MAX_ITER_VAL       = 255          # plenty of detail, fits [11:0]

# hold soft reset, clear stale status, then release
w(CTRL, CTRL_SOFT_RESET)
time.sleep(0.001)
w(CLR_STATUS, 0xF)                # clear all_done, axi_err, trans_count, debug
w(CTRL, 0)                        # release soft reset

w(FRACTAL_TYPE, FRACTAL_MANDELBROT)
w(CENTRE_X, 0)
w(CENTRE_Y, 0)
w(ZOOM_LEVEL, 0)
w(MAX_ITER, MAX_ITER_VAL)
w(IMAGE_BASE_ADDR, phys)          # bottom 8 bits ignored by RTL
w(IRQ_ENABLE, 0)                  # poll instead of IRQ

print("programmed:")
for name, off in [("FRACTAL_TYPE",FRACTAL_TYPE),("ZOOM",ZOOM_LEVEL),
                  ("MAX_ITER",MAX_ITER),("IMAGE_BASE",IMAGE_BASE_ADDR)]:
    print(f"  {name:12s} = 0x{r(off):08X}")

In [ ]:
# ── 6. Start + watch the writeback progress ──────────────────────────────────
w(CTRL, CTRL_START)               # self-clearing 1-cycle pulse

WORDS_PER_TILE = 16*16 // 8      # 32 for TILE_W=16
BURSTS_PER_TILE = WORDS_PER_TILE // 16  # 2 bursts per tile (16-beat AXI3 max)
TOTAL_TILES = 256
EXPECTED_BURSTS = TOTAL_TILES * BURSTS_PER_TILE  # 512

t0 = time.time()
TIMEOUT_S = 30
last = -1
while True:
    st    = r(STATUS)
    trans = r(TRANS_COUNT)
    if trans != last:
        b2d_raw = r(DBG_B2D_STATE)
        print(f"t={time.time()-t0:5.2f}s  STATUS=0x{st:02X}  "
              f"TRANS_COUNT={trans:5d}/{EXPECTED_BURSTS}  "
              f"tile_done_pop={r(DBG_TILE_DONE)}  "
              f"xfer_pop={r(DBG_XFER_POP)}  "
              f"b2d_fsm={b2d_raw&7}  axi_fsm={(b2d_raw>>5)&0xF}  "
              f"pending={( b2d_raw>>4)&1}  wr_ready={(b2d_raw>>9)&1}")
        last = trans
    if st & ST_ALL_DONE:
        print("all_done"); break
    if st & ST_AXI_ERR:
        print(f"!! AXI ERROR — BRESP last = 0b{r(DBG_BRESP_LAST)&3:02b}"); break
    if time.time() - t0 > TIMEOUT_S:
        print("!! TIMEOUT — engine/writeback did not finish"); break
    time.sleep(0.01)

trans_final = r(TRANS_COUNT)
aw = r(DBG_AW_HS_CNT)
w_hs = r(DBG_W_HS_CNT)
b_hs = r(DBG_B_HS_CNT)
b2d_raw = r(DBG_B2D_STATE)
print(f"\nfinal TRANS_COUNT      = {trans_final} / {EXPECTED_BURSTS} (missing {EXPECTED_BURSTS - trans_final})")
print(f"AW/W/B handshakes      = {aw} / {w_hs} / {b_hs}")
print(f"W beats per burst      = {w_hs // aw if aw else 'N/A'} (expect 16)")
print(f"DBG_FLAGS              = 0x{r(DBG_FLAGS):X}  (bit2=engine_done)")
print(f"b2d_fsm={b2d_raw&7}  axi_fsm={(b2d_raw>>5)&0xF}  pending={(b2d_raw>>4)&1}  wr_ready={(b2d_raw>>9)&1}")
print(f"BRESP last             = 0b{r(DBG_BRESP_LAST)&3:02b}")

In [ ]:
# ── 7. Sanity-check what actually landed in DDR ──────────────────────────────
img_buf.invalidate()              # pull fresh data back from DRAM into the view
nonzero = int(np.count_nonzero(img_buf))
print(f"non-zero bytes in DDR buffer = {nonzero} / {IMAGE_BYTES} "
      f"({100*nonzero/IMAGE_BYTES:.1f}%)")
if nonzero == 0:
    print("  -> nothing written. Check TRANS_COUNT above and HP port/clock.")

In [ ]:
# ── 8. Reconstruct + display the image ───────────────────────────────────────
# DDR layout (per the RTL writeback): the image is 4x4 sixteenths of 256x256.
# Each sixteenth is written tile-by-tile; within a sixteenth the address is
#   off = sixteenth*65536 + tile_idx*BYTES_PER_TILE + word*8 (+byte)
# For a quick view we treat each sixteenth as a contiguous 256x256 block laid out
# in tile order, then place the 16 sixteenths into a 4x4 grid.
import matplotlib.pyplot as plt

TILE_W   = 16
TPA      = SIXTEENTH_W // TILE_W            # tiles per axis within a sixteenth (16)
GRID     = 4                                # 4x4 sixteenths -> 1024x1024

def sixteenth_to_image(flat):
    """flat: 65536 bytes in (tile_row, tile_col, row_in_tile, col_in_tile) order."""
    out = np.zeros((SIXTEENTH_W, SIXTEENTH_W), np.uint8)
    idx = 0
    for tr in range(TPA):
        for tc in range(TPA):
            blk = flat[idx:idx+TILE_W*TILE_W].reshape(TILE_W, TILE_W)
            out[tr*TILE_W:(tr+1)*TILE_W, tc*TILE_W:(tc+1)*TILE_W] = blk
            idx += TILE_W*TILE_W
    return out

full = np.zeros((GRID*SIXTEENTH_W, GRID*SIXTEENTH_W), np.uint8)
data = np.asarray(img_buf)
for s in range(SIXTEENTHS):
    seg = data[s*65536:(s+1)*65536]
    if seg.size < 65536:
        break
    sx = sixteenth_to_image(seg)
    gr, gc = s // GRID, s % GRID
    full[gr*SIXTEENTH_W:(gr+1)*SIXTEENTH_W, gc*SIXTEENTH_W:(gc+1)*SIXTEENTH_W] = sx

plt.figure(figsize=(8, 8))
plt.imshow(full, cmap='twilight_shifted', vmin=0, vmax=63)
plt.title('Mandelbrot from DDR writeback'); plt.axis('off'); plt.show()

In [ ]:
# ── 9. (optional) free the buffer when done ──────────────────────────────────
# del img_buf   # releases the contiguous allocation